# 06 · Build the modeling table: features at hour t → label at hour t+1
One row per airport per UTC hour *t*. Column prefixes:
- `feat_*`: known at the end of hour t, so safe to use as model inputs.
  - The **t+1 schedule** (`feat_sched_*_next`, `feat_expected_runway_next`) is included because airline schedules are published months ahead.
  - **`feat_runway_demand_next`** = backlog at the end of t (leftover flights) + flights the schedule puts on the runway in t+1. This is the project's demand measure.
- `wxnext_*`: *observed* weather at t+1, a stand-in for a perfect forecast. Kept separate so each model can include or exclude it.
- `label_over_capacity_next`: the target, i.e. whether demand exceeded capacity in hour t+1.
- **Longer horizons** (`HORIZONS` in config): `label_over_capacity_h3` / `_h6` are the target at t+3 / t+6.
  - The schedule is known in advance, so `feat_expected_runway_h{H}`, `feat_sched_total_h{H}` and `feat_expected_runway_sum_h{H}` (flights expected over t+1…t+H) are legitimate features.
  - `wxh{H}_*` is the observed weather at t+H, a stand-in for a forecast that far ahead.
- `diag_next_*`: the ingredients of that label (for checking how robust thresholds are). **Never use these as features.**
- `meta_*`: identifiers and split.

Output: `data/processed/{APT}_hourly.parquet`

In [1]:
%run ../config.ipynb
import holidays

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → latest | train ≤ 2025-12


In [2]:
def load(apt):
    fl = pd.read_parquet(INTERIM / f"hourly_flights_{apt}.parquet")
    wx = pd.read_parquet(INTERIM / f"weather_{apt}.parquet").drop(columns="airport")
    rw = pd.read_parquet(INTERIM / f"runway_wind_{apt}.parquet").drop(columns="airport")
    df = fl.merge(wx, on="hour_utc", how="left").merge(rw, on="hour_utc", how="left")
    assert len(df) == len(fl), "join changed the row count"
    return df

## Label: over capacity in hour h
Either of the following:
- share of scheduled arrivals with NAS + weather delay ≥ 15 min is at least `NAS_WX_SHARE_MIN` (only when at least `MIN_SCHED_ARR_FOR_SHARE` arrivals are scheduled)
- weather/NAS cancellations ≥ `CANCEL_WX_NAS_MIN`

Backlog growth was dropped from the label (see config); it stays as a feature and as `diag_next_backlog_growth`.

In [3]:
def label_parts(df):
    share = (df["arr_nas_wx_delayed"] / df["sched_arr"]).where(df["sched_arr"] >= MIN_SCHED_ARR_FOR_SHARE, 0.0)
    return pd.DataFrame({"nas_wx_share": share.fillna(0),
                         "backlog_growth": df["backlog_growth"],
                         "cancel_wx_nas": df["cancel_wx_nas"],
                         "taxi_out_excess": df["taxi_out_excess"]})


def over_capacity(parts, share, cancel):
    return ((parts["nas_wx_share"] >= share) | (parts["cancel_wx_nas"] >= cancel)).astype(int)

## Features at hour t

In [4]:
WX_COLS = ["temp_c", "dewpoint_c", "slp_hpa", "wind_kt", "gust_kt", "wind_max_kt", "wind_u", "wind_v",
           "wind_variable", "vis_sm", "vis_min_sm", "ceiling_ft", "ceiling_min_ft", "precip_mm",
           "flight_cat", "imc", "n_specials", "wx_missing",
           "wx_ts", "wx_snow", "wx_freezing", "wx_fog", "wx_rain", "wx_heavy"]
RW_COLS = ["n_usable_ends", "n_usable_ils_ends", "n_usable_arrival_ends", "frac_usable",
           "n_usable_physical", "n_usable_headings", "min_crosswind_kt", "max_headwind_kt", "crosswind_on_best_kt"]
OPS_COLS = ["sched_dep", "sched_arr", "sched_total", "actual_dep", "actual_arr", "actual_total",
            "cancel_dep", "cancel_arr", "cancel_wx_nas", "diverted_arr", "arr_nas_wx_delayed",
            "arr_del15", "dep_del15", "arr_delay_mean", "dep_delay_mean",
            "taxi_out_mean", "taxi_in_mean", "taxi_out_excess",
            "backlog", "backlog_dep", "backlog_arr", "backlog_roll4", "backlog_growth"]
LAG_COLS = ["backlog", "backlog_growth", "nas_wx_share", "over_capacity", "actual_total", "imc", "wind_kt"]
NEXT_SCHED = ["sched_dep", "sched_arr", "sched_total"]
SCHEDULE_KNOWN = ["sched_dep", "sched_arr", "sched_total", "expected_dep_all", "expected_arr_all", "expected_total_all"]
WXNEXT_COLS = ["wind_kt", "gust_kt", "vis_sm", "ceiling_ft", "precip_mm", "imc", "flight_cat",
               "wx_ts", "wx_snow", "wx_freezing", "n_usable_ends", "n_usable_ils_ends", "n_usable_arrival_ends"]


def build(apt):
    df = load(apt).sort_values("hour_utc").reset_index(drop=True)
    assert df["hour_utc"].diff().dropna().eq(pd.Timedelta("1h")).all(), "spine must be continuous for shift()"
    parts = label_parts(df)
    df["nas_wx_share"] = parts["nas_wx_share"]
    df["over_capacity"] = over_capacity(parts, NAS_WX_SHARE_MIN, CANCEL_WX_NAS_MIN)

    out = pd.DataFrame({"meta_airport": apt, "meta_hour_utc": df["hour_utc"],
                        "meta_local_time": df["local_time"]})

    # --- features observed by the end of hour t
    cur = {}
    for c in WX_COLS + RW_COLS + OPS_COLS + ["nas_wx_share", "over_capacity"]:
        cur[f"feat_{c}"] = df[c]
    for c in LAG_COLS:
        for k in (1, 2):
            cur[f"feat_{c}_lag{k}"] = df[c].shift(k)
    cur["feat_precip_3h"] = df["precip_mm"].rolling(3, min_periods=1).sum()
    cur["feat_over_capacity_last6"] = df["over_capacity"].rolling(6, min_periods=1).sum()

    # Demonstrated capacity: 95th percentile of hourly runway throughput over the previous 28 days
    # (closed='left' leaves out the current hour). Picks up closures and FAA caps automatically.
    thr = df.set_index("hour_utc")["actual_total"]
    demo = thr.rolling(DEMONSTRATED_WINDOW, closed="left", min_periods=24 * 7).quantile(DEMONSTRATED_Q)
    cur["feat_demonstrated_capacity"] = demo.to_numpy()

    # Calendar (local time)
    lt = df["local_time"]
    us_hol = holidays.country_holidays("US", years=range(lt.dt.year.min(), lt.dt.year.max() + 1))
    cur["feat_hour_local"] = lt.dt.hour
    cur["feat_hour_sin"] = np.sin(2 * np.pi * lt.dt.hour / 24)
    cur["feat_hour_cos"] = np.cos(2 * np.pi * lt.dt.hour / 24)
    cur["feat_dow"] = lt.dt.day_of_week
    cur["feat_month"] = lt.dt.month
    cur["feat_holiday"] = lt.dt.date.map(lambda d: d in us_hol).astype(int)

    # --- t+1 schedule (known in advance)
    nxt = {f"feat_{c}_next": df[c].shift(-1) for c in NEXT_SCHED}
    # Demand for t+1: leftover flights at the end of t + flights scheduled to reach the runway in t+1.
    # Cancellations stay in the t+1 count, since at the end of hour t we don't reliably know which ones will be cancelled.
    nxt["feat_expected_runway_next"] = df["expected_total_all"].shift(-1)
    nxt["feat_runway_demand_next"] = df["backlog"] + nxt["feat_expected_runway_next"]
    nxt["feat_demand_vs_demonstrated"] = nxt["feat_runway_demand_next"] / cur["feat_demonstrated_capacity"]

    # --- longer horizons: schedule at t+H (known in advance), weather at t+H (forecast stand-in), label at t+H
    wxn = {f"wxnext_{c}": df[c].shift(-1) for c in WXNEXT_COLS}
    lab = {"label_over_capacity_next": df["over_capacity"].shift(-1)}
    meta_h = {}
    for H in [h for h in HORIZONS if h > 1]:
        nxt[f"feat_expected_runway_h{H}"] = df["expected_total_all"].shift(-H)
        nxt[f"feat_sched_total_h{H}"] = df["sched_total"].shift(-H)
        nxt[f"feat_expected_runway_sum_h{H}"] = sum(df["expected_total_all"].shift(-k) for k in range(1, H + 1))
        for c in WXNEXT_COLS:
            wxn[f"wxh{H}_{c}"] = df[c].shift(-H)
        lab[f"label_over_capacity_h{H}"] = df["over_capacity"].shift(-H)
        meta_h[f"meta_active_h{H}"] = (df["sched_total"].shift(-H) > 0).astype(int)
    for name, th in LABEL_SENSITIVITY.items():
        lab[f"diag_next_label_{name}"] = over_capacity(parts, **th).shift(-1)
    for c in parts:
        lab[f"diag_next_{c}"] = parts[c].shift(-1)

    out = pd.concat([out, pd.DataFrame(cur), pd.DataFrame(nxt), pd.DataFrame(wxn), pd.DataFrame(lab),
                     pd.DataFrame(meta_h)], axis=1)
    out = out.iloc[:-max(HORIZONS)]           # the last hours have no t+H
    train_end = pd.Period(TRAIN_END, "M").end_time
    out["meta_split"] = np.where(out["meta_local_time"] <= train_end, "train", "test")
    out["meta_active"] = (out["feat_sched_total_next"] > 0).astype(int)   # t+1 has scheduled flights
    for c in [c for c in out if c.startswith("label_")]:
        out[c] = out[c].astype(int)
    return out

## Leakage guard
The only `feat_` columns allowed to use data from after hour t are the future-schedule columns, which are known in advance. Check this by construction: rebuild with every *outcome* after hour t scrambled (all horizons), and confirm that no other `feat_` column changes.

In [5]:
def leakage_check(apt):
    base = build(apt)
    orig_load = load

    def scrambled(a):
        df = orig_load(a)
        rng = np.random.default_rng(0)
        future = df.index[-max(HORIZONS):]          # every hour after the last row's hour t
        outcome_cols = [c for c in df if c not in ["hour_utc", "local_time", "airport"] + SCHEDULE_KNOWN]
        donors = rng.choice(df.index[:-max(HORIZONS) - 24], len(future))
        df.loc[future, outcome_cols] = df.loc[donors, outcome_cols].to_numpy()
        return df

    globals()["load"] = scrambled
    try:
        pert = build(apt)
    finally:
        globals()["load"] = orig_load
    # Outcomes after the final row's hour changed; that row's feat_ columns must not change.
    i = len(base) - 1
    diag = [c for c in base if c.startswith(("diag_", "label_", "wxnext_", "wxh"))]
    assert any(base.at[i, c] != pert.at[i, c] for c in diag), "scramble had no effect, so the test is vacuous"
    feats = [c for c in base if c.startswith("feat_")]
    changed = [c for c in feats if not (pd.isna(base.at[i, c]) and pd.isna(pert.at[i, c]))
               and base.at[i, c] != pert.at[i, c]]
    allowed = {f"feat_{c}_next" for c in NEXT_SCHED} | {"feat_expected_runway_next", "feat_runway_demand_next",
                                                         "feat_demand_vs_demonstrated"} \
        | {c for c in feats if c.rsplit("_", 1)[-1] in {f"h{H}" for H in HORIZONS}}
    assert set(changed) <= allowed, f"leakage: {set(changed) - allowed}"
    print(f"leakage check passed ({len(feats)} feat_ columns; t+1-dependent: {sorted(changed) or 'none changed'})")

In [6]:
datasets = {}
for apt in STUDY_AIRPORTS:
    ds = build(apt)
    leakage_check(apt)
    datasets[apt] = ds
    ds.to_parquet(PROCESSED / f"{apt}_hourly.parquet", index=False)
    print(f"{apt}: {len(ds):,} rows × {ds.shape[1]} columns → {PROCESSED / f'{apt}_hourly.parquet'}")

leakage check passed (93 feat_ columns; t+1-dependent: none changed)
EWR: 32,849 rows × 150 columns → /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final/data/processed/EWR_hourly.parquet


## Checks

In [7]:
for apt, ds in datasets.items():
    act = ds[ds["meta_active"] == 1]
    print(f"== {apt}: {len(ds):,} hours, {len(act):,} with scheduled flights in t+1")
    print(ds.groupby("meta_split")["label_over_capacity_next"].agg(["size", "mean"]).round(3))
    sens = act[[c for c in act if c.startswith("diag_next_label_")]].mean().round(3)
    print("positive rate (active hours) by label setting:\n", sens.to_string())
    by_month = act.groupby(act["meta_local_time"].dt.to_period("M"))["label_over_capacity_next"].mean()
    display(by_month.round(3).to_frame("pos_rate").T)
    trig = act[act["label_over_capacity_next"] == 1]
    print("which rule fired (positives):",
          {"nas_wx_share": (trig["diag_next_nas_wx_share"] >= NAS_WX_SHARE_MIN).mean().round(3),
           "cancel_wx_nas": (trig["diag_next_cancel_wx_nas"] >= CANCEL_WX_NAS_MIN).mean().round(3)})
    print("feature missingness (top 8):")
    print(act.filter(like="feat_").isna().mean().sort_values(ascending=False).head(8).round(3).to_string())

== EWR: 32,849 hours, 30,034 with scheduled flights in t+1
             size   mean
meta_split              
test         6545  0.139
train       26304  0.157
positive rate (active hours) by label setting:
 diag_next_label_strict      0.111
diag_next_label_default     0.168
diag_next_label_loose       0.247
diag_next_label_nas_only    0.137


meta_local_time,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
pos_rate,0.166,0.144,0.112,0.191,0.079,0.306,0.371,0.078,0.197,0.009,0.009,0.035,0.087,0.059,0.045,0.103,0.096,0.203,0.247,0.27,0.071,0.12,0.253,0.168,0.128,0.104,0.139,0.274,0.492,0.198,0.386,0.226,0.12,0.222,0.25,0.193,0.147,0.166,0.166,0.041,0.041,0.135,0.305,0.22,0.139


which rule fired (positives): {'nas_wx_share': np.float64(0.82), 'cancel_wx_nas': np.float64(0.378)}
feature missingness (top 8):
feat_dep_delay_mean            0.177
feat_taxi_out_excess           0.119
feat_taxi_out_mean             0.119
feat_arr_delay_mean            0.050
feat_taxi_in_mean              0.018
feat_demand_vs_demonstrated    0.005
feat_demonstrated_capacity     0.005
feat_wind_u                    0.002


## Sanity check: does the label behave like a capacity limit?
If labeled hours really are hours when demand exceeded capacity, then, compared with unlabeled hours, they should show:
1. **Throughput below demand.** The airport didn't get through what was scheduled (`actual / expected`).
2. **Throughput that levels off.** As scheduled demand rises, unlabeled hours keep up, while labeled hours stall below the line.
3. **Lower throughput at the same demand.** Within the same demand bin, labeled hours handle fewer flights.
4. **More capacity-reducing weather:** instrument conditions (IMC), thunderstorms, fewer usable arrival runways.
5. **Queue symptoms the label never looked at:** excess taxi-out time and growing backlog.
6. **Delay blamed on the airspace system rather than the airline**, in the BTS delay-cause minutes.

All of these use hour *t*'s own values (`feat_over_capacity` is the label at t).

In [8]:
fl_cols = ["DEST", "sched_arr_utc", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "LATE_AIRCRAFT_DELAY"]
for apt, ds in datasets.items():
    hf = pd.read_parquet(INTERIM / f"hourly_flights_{apt}.parquet")[["hour_utc", "expected_dep_nc", "expected_arr_nc"]]
    a = ds[ds["meta_active"] == 1].merge(hf, left_on="meta_hour_utc", right_on="hour_utc", how="left")
    a["expected"] = a["expected_dep_nc"] + a["expected_arr_nc"]
    a["label"] = a["feat_over_capacity"].map({1: "over capacity", 0: "not flagged"})
    print(f"== {apt}: label rate (active hours, hour t) = {a['feat_over_capacity'].mean():.3f}")

    # 1, 4, 5: compare means
    summary = a.groupby("label").agg(
        hours=("label", "size"),
        scheduled=("feat_sched_total", "mean"),
        expected_at_runway=("expected", "mean"),
        actual_throughput=("feat_actual_total", "mean"),
        demonstrated_capacity=("feat_demonstrated_capacity", "mean"),
        imc_share=("feat_imc", "mean"),
        thunderstorm_share=("feat_wx_ts", "mean"),
        usable_arrival_ends=("feat_n_usable_arrival_ends", "mean"),
        taxi_out_excess_min=("feat_taxi_out_excess", "mean"),
        backlog_growth=("feat_backlog_growth", "mean"),
        arr_delay_min=("feat_arr_delay_mean", "mean"))
    summary["throughput_vs_expected"] = summary["actual_throughput"] / summary["expected_at_runway"]
    display(summary.T.round(2))

    # 2, 3: throughput by demand bin
    a["demand_bin"] = pd.cut(a["expected"], [0, 20, 30, 40, 50, 60, 200], right=False)
    curve = a.pivot_table(index="demand_bin", columns="label", values="feat_actual_total",
                          aggfunc=["mean", "size"], observed=True).round(1)
    print("throughput (wheels-off + wheels-on) by flights expected at the runway that hour:")
    display(curve)

    # 6: delay-cause minute mix
    fl = pd.read_parquet(INTERIM / f"flights_{apt}.parquet", columns=fl_cols)
    fl = fl[fl["DEST"] == apt].assign(hour_utc=lambda x: x["sched_arr_utc"].dt.floor("h"))
    fl = fl.merge(a[["meta_hour_utc", "label"]], left_on="hour_utc", right_on="meta_hour_utc")
    causes = ["CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "LATE_AIRCRAFT_DELAY"]
    mix = fl.groupby("label")[causes].sum()
    print("share of arrival delay minutes by BTS cause:")
    display((mix.div(mix.sum(axis=1), axis=0) * 100).round(1))

== EWR: label rate (active hours, hour t) = 0.164


label,not flagged,over capacity
hours,25094.00,4940.00
scheduled,30.79,36.32
expected_at_runway,30.54,31.85
actual_throughput,30.54,30.52
demonstrated_capacity,46.54,45.41
imc_share,0.07,0.15
thunderstorm_share,0.00,0.05
usable_arrival_ends,4.27,3.87
taxi_out_excess_min,1.00,12.22
backlog_growth,0.00,1.33


throughput (wheels-off + wheels-on) by flights expected at the runway that hour:


mean                      size              
label      not flagged over capacity not flagged over capacity
demand_bin                                                    
[0, 20)            8.1          17.1        6200           831
[20, 30)          28.5          26.4        2415           983
[30, 40)          35.7          32.3        8290          1751
[40, 50)          41.9          38.1        6791          1114
[50, 60)          47.8          44.7        1356           246
[60, 200)         53.6          48.5          42            15

share of arrival delay minutes by BTS cause:


,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,LATE_AIRCRAFT_DELAY
label,,,,
not flagged,37.8,3.6,17.1,41.5
over capacity,12.5,5.8,62.7,19.1
